The notebook is a Proof of concept of summarization of politic speeches.
It evaluates 4 models (For complet benchmark, we have to use more than 1 speech) 

In [ ]:
from datetime import datetime
import hashlib
import json
from neo4j import GraphDatabase
import ollama
import os
import pandas as pd
from sentence_transformers import SentenceTransformer
import sys
import time
from tqdm import tqdm
import torch
sys.path.insert(1, "src/graph/")
from graph_builder import (
    get_node_id,
    extract_graph,
    compute_chunk_embeddings,
    merge_graphs,
    validate_graph,
    add_speaker_entities,
    build_neo4j_graph,
    feed_global_report,
    save_graph,
    load_to_neo4j,
    create_constraints
)
sys.path.insert(1, "src/preprocessing/")
from speeches import (
    load_speeches,
    split_into_chunks,
    load_prompt_template
)


# Goal

The goal of the pipeline is to extract information inside a document. For the demonstration we use public speech as dataset:

 ```src/test/2017-05-14_d-claration-de-m-emmanuel-macron-pr-sident-de-la-r-publique.txt```

The extraction step will be processed by a LLM. Different models exist and the result depend on many criteria:

* Model should be multi-langage or French
* Can handle abstract concept 
* Can be executed on a home desktop in a efficient way

According to the litterature ```qwen2.5:7b``` model fit those criteria and have good results in text extraction.

# Install Ollama and the Qwen model

Before using the pipeline, install the following dependencies.

**Ollama** on Windows the install command is:

```
irm https://ollama.com/install.ps1 | iex
```

Check that the Ollama server is running:

```
ollama list
```

You can also check the API response directly:

```
curl http://localhost:11434/api/tags
```

If you've just installed Ollama, you won't have any models yet. Pull `qwen2.5:7b`:

```
ollama pull qwen2.5:7b
```

Check that it runs correctly:

```
ollama run qwen2.5:7b
```

You can type anything into the prompt to confirm it responds.

# Load speech

We load the speech and made a short analysis about it

As you can see, the file is composed of many bloc. Title, date, source, speech, keywords...
The speech is between the keywords:

* Texte intégral
* MOTS CLÉS

We will extract the speech and keep the other information. For this purpose we will use a home made function.
The function will return a list of dictionary. As we have only 1 text, we will save the dict intto ```speech```.

Here the different keys field:

* id
* date
* title
* filename
* header
* text
* speakers
* chunks

In [ ]:
corpus = load_speeches(
    os.path.join(
        "src", "test", "speech")
    )

speech = corpus[0]

In [ ]:
print(speech["text"])

# Summarization 

As you can see, the document is the first speech of the French president Macron.
Before extract information about it, lets do premilary analysis by doing summarization.
It creates a shorter version of a document or an article that captures all the important information.

We use Encoder-Decoder model. First step is to split the speech on different chunks. Indeed the speech is too long for our current model.
The function ```split_into_chunks()``` call ```RecursiveCharacterTextSplitter``` from ```langchain_text_splitters```.

**In the current step, we set the chunk_overlap to 0 because we want to summarize. In RAG context, we will change the value** 


In [ ]:
chunks = split_into_chunks(speech, chunk_size=1200, chunk_overlap=0)

In [ ]:
chunks[1]

### Benchmarking models
chunk_overlap is set 0 and split speech as chunks. This vector will be use as input for the transformer to summarize the text.
As we use french text, we need a multilangual or French model. A benchmark is made with these models:  

* ```mT5_multilingual_XLSum```
* ```BARThez```
* ```CamemBERT```
* ```FlauBERT```

For that goal, lets create a function takes a instantiated tokenizer, model and prompts (instruction + chunk).

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

@torch.inference_mode()
def summarize_batch_benchmark(tokenizer : AutoTokenizer, 
                              model: AutoModelForSeq2SeqLM, 
                              prompts: list[str], 
                              device: torch.device | str,
                              max_new_tokens = 100,
                              forced_bos_token_id: int | None = None,
                              use_bfloat16: bool = False):
    """
    Summarize a piece of text using a Transformer summarization model.

    Args:
        tokenizer (AutoTokenizer): Instantiated tokenizer
        model (AutoModelForSeq2SeqLM): Instantiated model
        prompts (list[str]): Instruction and Text to summarize
        max_new_tokens (int): Maximum number of tokens generated for the summary.

    Returns:
        str:
            Generated summary.
    """
    # summaries = []
    inputs = tokenizer(
    prompts ,
    return_tensors="pt",
    truncation=True,
    padding=True, #set vector to the same size for the batching
    max_length=1024
    )
    # send data into GPU or CPU
    inputs = {k: v.to(device) for k, v in inputs.items()}

    start_time = time.perf_counter()

    # Inference
    with torch.amp.autocast(enabled=use_bfloat16, dtype=torch.bfloat16, device_type = device):
    #unable gradient compute and storage (efficient for RAM)
    # with torch.no_grad():
        if forced_bos_token_id is None:
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                num_beams =4,
                do_sample = False,
                no_repeat_ngram_size = 3
            )
        else:
            outputs = model.generate(
                **inputs,
                forced_bos_token_id=forced_bos_token_id,
                max_new_tokens=max_new_tokens,
                num_beams =4,
                do_sample = False,
                no_repeat_ngram_size = 3
            )
    # Benchmark metric 
    execution_time = time.perf_counter() - start_time
    generated_tokens_count = outputs.numel()
    if execution_time > 0 :
        tokens_per_second = generated_tokens_count / execution_time
    else:
        tokens_per_second = 0
    metrics = {
        "execution_time_sec": round(execution_time, 4),
        "total_generated_tokens": generated_tokens_count,
        "tokens_per_second": round(tokens_per_second, 2),
        "batch_size": len(prompts)
    }
    # print(tokenizer.decode(outputs[0], skip_special_tokens=True))
    return tokenizer.batch_decode(outputs, skip_special_tokens=True), metrics
    # summaries.extend(batch_summaries)
    # return summaries

#### mT5_multilingual_XLSum

In [ ]:
#from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
#import sentencepiece as spm

checkpoint = "csebuetnlp/mT5_multilingual_XLSum" 
# load a T5 tokenizer to process text and summary
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Calculs exécutés sur : {device.upper()}")
chunk_summaries_mT5, metrics_mT5 = summarize_batch_benchmark(
    tokenizer=tokenizer,
    model=model,
    prompts = ["summarize: " + c["text"] for c in chunks],
    device=device
)

In [ ]:
#Let see the first 5 chunks and the summary
for i in range(5):
    print("=" * 100)
    print(f"CHUNK {i}")
    print(chunks[i]["text"])
    print("\nSUMMARY:")
    print(chunk_summaries_mT5[i])

As you can see. The model cannot summary the different chunk and hallucinate.

#### BARThez

In [ ]:
from transformers import BartTokenizer, AutoModelForSeq2SeqLM

barthez_ckpt = "moussaKam/barthez-orangesum-abstract"

# Chargement du tokenizer et du modèle appropriés
barthez_tokenizer = BartTokenizer.from_pretrained(barthez_ckpt)
barthez_model = AutoModelForSeq2SeqLM.from_pretrained(barthez_ckpt)

if barthez_tokenizer.pad_token is None:
    barthez_tokenizer.pad_token = barthez_tokenizer.eos_token

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Calculs exécutés sur : {device.upper()}")
chunk_summaries_BARThez, metrics_BARThez = summarize_batch_benchmark(
    tokenizer = barthez_tokenizer,
    model = barthez_model,
    prompts = [c["text"] for c in chunks], #BARThez does not need a "summarize: " prefix like T5
    device=device
)

Let see the first 5 chunks and the summary

In [ ]:
for i in range(5):
    print("=" * 100)
    print(f"CHUNK {i}")
    print(chunks[i]["text"])
    print("\nSUMMARY:")
    print(chunk_summaries_BARThez[i])

#### CamemBERT

In [ ]:
from transformers import MBart50TokenizerFast, AutoModelForSeq2SeqLM

camembert_ckpt = "facebook/mbart-large-50-many-to-many-mmt"

camembert_tokenizer = MBart50TokenizerFast.from_pretrained(camembert_ckpt)
camembert_model = AutoModelForSeq2SeqLM.from_pretrained(camembert_ckpt)

# Configuration de la langue cible en Français pour forcer le modèle à répondre en français
camembert_tokenizer.src_lang = "fr_XX"
# On récupère l'identifiant du jeton français pour le donner au décodeur
forced_bos_token_id = camembert_tokenizer.lang_code_to_id["fr_XX"]

if camembert_tokenizer.pad_token is None:
    camembert_tokenizer.pad_token = camembert_tokenizer.eos_token

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Calculs exécutés sur : {device.upper()}")
chunk_summaries_camembert, metrics_camembert = summarize_batch_benchmark(
    tokenizer = camembert_tokenizer,
    model = camembert_model,
    prompts = [c["text"] for c in chunks],
    device=device,
    forced_bos_token_id=forced_bos_token_id
)



Let see the first 5 chunks and the summary

In [ ]:
for i in range(5):
    print("=" * 100)
    print(f"CHUNK {i}")
    print(chunks[i]["text"])
    print("\nSUMMARY:")
    print(chunk_summaries_camembert[i])

#### FlauBERT

In [ ]:
from transformers import T5TokenizerFast, AutoModelForSeq2SeqLM

flaubert_ckpt = "plguillou/t5-base-fr-sum-cnndm"

flaubert_tokenizer = T5TokenizerFast.from_pretrained(flaubert_ckpt)
flaubert_model = AutoModelForSeq2SeqLM.from_pretrained(flaubert_ckpt)

if flaubert_tokenizer.pad_token is None:
    flaubert_tokenizer.pad_token = flaubert_tokenizer.eos_token

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Calculs exécutés sur : {device.upper()}")
chunk_summaries_flaubert, metrics_flaubert = summarize_batch_benchmark(
    tokenizer = flaubert_tokenizer,
    model = flaubert_model,
    prompts = ["summarize: "  + c["text"] for c in chunks], 
    device=device
)

Let see the first 5 chunks and the summary

In [ ]:
for i in range(5):
    print("=" * 100)
    print(f"CHUNK {i}")
    print(chunks[i]["text"])
    print("\nSUMMARY:")
    print(chunk_summaries_flaubert[i])

#### Model performance

* mT5 hallucinate 
* BARThez's output format is not correct
* Camembert output is troncated. The model does not summarize correctly 
* flaubert seems to give correct results

In case we have 2 or more model with good results, we can compare the number of token per seconde. Choose the most efficient is important to increase the speed process.
A lot of time is lost during the inference process (we wait the end of the procedure).

In [ ]:
pd.DataFrame(
    {    
        'model' : ["mT5", "BARThez", "camembert", "flaubert" ],
        'tokens_per_second' : [metrics_mT5['tokens_per_second'], metrics_BARThez['tokens_per_second'], metrics_camembert['tokens_per_second'], metrics_flaubert['tokens_per_second'] ]
        
    }
)